# RQL highway evaluation on Colab

This notebook clones a selected Git ref, creates an isolated virtual environment, runs `eval_highway.py`, and saves the complete evaluation log plus run parameters to Google Drive.

Ensure the selected Git ref contains `eval_highway.py` and the model selected below. Use a distinct `RUN_NAME` for parallel Colab sessions.

In [ ]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Experiment parameters (edit this cell before running).
from datetime import datetime, timezone

REPO_URL = "https://github.com/thowell332/RQL-Comparison.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
SUPERVISOR_REPO_URL = "https://github.com/phbui/norm-supervised-highway.git"  #@param {type:"string"}
SUPERVISOR_REPO_REF = "main"  #@param {type:"string"}

# Paths may be repository-relative or absolute (including paths on Drive).
MODEL_PATH = "logs/highway-ME-basic-v0_Example_Pretrained.zip"  #@param {type:"string"}
ENV_NAME = "highway-ME-basic-AddRightRewardALL-v0"  #@param {type:"string"}
N_EPISODES = 1000  #@param {type:"integer"}
N_STEPS = None  # Used only when N_EPISODES is None; script default is 400.
SEED = 239  #@param {type:"integer"}
MODEL_TYPE = "auto"  #@param ["auto", "dqn_me", "residual"]
SAFE_DECIDE = False  #@param {type:"boolean"}

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")  #@param {type:"string"}
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/aaai2027/RQL-Comparison/eval_highway"  #@param {type:"string"}

In [ ]:
# Clone the requested ref and create a Colab-compatible evaluation environment.
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/RQL-Comparison")
SUPERVISOR_DIR = Path("/content/norm-supervised-highway-dependency")
VENV_DIR = Path("/content/venvs/rql-comparison")

for path in (PROJECT_DIR, SUPERVISOR_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)
subprocess.run(["git", "clone", SUPERVISOR_REPO_URL, str(SUPERVISOR_DIR)], check=True)
subprocess.run(["git", "-C", str(SUPERVISOR_DIR), "checkout", SUPERVISOR_REPO_REF], check=True)
# Reuse Colab's CUDA-enabled PyTorch while isolating experiment-specific packages.
subprocess.run(["python3", "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)
VENV_PYTHON = VENV_DIR / "bin/python"

subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
# The repository-wide requirements pin Python-3.9-era training packages. These are
# the complete dependencies needed by the highway evaluation path on current Colab.
EVAL_PACKAGES = [
    "numpy<2", "gym==0.23.1", "scipy", "pandas", "cloudpickle",
    "matplotlib", "tensorboard", "tensorboardX", "pygame", "tqdm",
    "rich", "pyyaml",
]
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", *EVAL_PACKAGES], check=True)
# eval_highway.py imports this package, but it is maintained in the companion repo.
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-deps", "-e", str(SUPERVISOR_DIR)],
    check=True,
)

entry_point = PROJECT_DIR / "eval_highway.py"
if not entry_point.is_file():
    raise FileNotFoundError(f"Selected Git ref does not contain {entry_point}")
print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")

In [ ]:
# Run eval_highway.py and stream its complete output to both Colab and Drive.
import json
import os

RUN_DIR = Path(DRIVE_RESULTS_ROOT).expanduser() / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

def resolve_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_DIR / path

model_path = resolve_path(MODEL_PATH)
if not model_path.exists():
    raise FileNotFoundError(f"Model not found: {model_path}. Commit it to the selected Git ref or use an absolute Drive path.")

manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "supervisor_repo_url": SUPERVISOR_REPO_URL,
    "supervisor_repo_ref": SUPERVISOR_REPO_REF,
    "model_path": str(model_path),
    "env_name": ENV_NAME,
    "n_episodes": N_EPISODES,
    "n_steps": N_STEPS,
    "seed": SEED,
    "model_type": MODEL_TYPE,
    "safe_decide": SAFE_DECIDE,
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))

command = [
    str(VENV_PYTHON), str(PROJECT_DIR / "eval_highway.py"),
    "--model-path", str(model_path),
    "--env-name", ENV_NAME,
    "--seed", str(SEED),
    "--model-type", MODEL_TYPE,
]
if N_EPISODES is not None:
    command += ["--n-episodes", str(N_EPISODES)]
elif N_STEPS is not None:
    command += ["--n-steps", str(N_STEPS)]
if SAFE_DECIDE:
    command.append("--safe-decide")

print("$", " ".join(command))
with (RUN_DIR / "evaluation.log").open("w", buffering=1) as log:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

print(f"Results saved to: {RUN_DIR}")

In [ ]:
# Verify the artifacts persisted to Drive.
for artifact in sorted(path for path in RUN_DIR.rglob("*") if path.is_file()):
    print(f"{artifact.relative_to(RUN_DIR)}  ({artifact.stat().st_size:,} bytes)")